### 1. Import Libraries

This cell imports the necessary libraries for the script, including `pandas` for data manipulation, `docx-template` for working with Word templates, `os` for file system operations, and `re` for regular expressions.

In [64]:
import pandas as pd
import re
import os
from docxtpl import DocxTemplate, RichText

# Data Formatting Class

This notebook processes data from a CSV file and generates Word documents based on a template. To make the code more organized, reusable, and easier to maintain, we will define a `DataFormatter` class. This class will encapsulate all the formatting logic for various data fields.

The formatting functions will handle:
- **Full Name**: Converting to uppercase.
- **Date of Birth**: Changing the format from `m/d/yyyy` to `dd/mm/yyyy`.
- **Phone Number**: Ensuring it's a string and starts with '0'.
- **Specialized Board**: Converting boolean-like values into checkbox symbols for the document.

In [65]:
def format_ho_ten(name):
    """Converts name to uppercase."""
    return str(name).upper()

In [66]:
def format_ngay_sinh(date_str):
    """Formats a date string from m/d/yyyy to dd/mm/yyyy."""
    try:
        # Attempt to parse with pandas to handle various formats gracefully
        date_obj = pd.to_datetime(date_str)
        return date_obj.strftime('%d/%m/%Y')
    except Exception:
        # Fallback for unexpected formats
        try:
            day, month, year = date_str.split('/')
            return f"{day.zfill(2)}/{month.zfill(2)}/{year}"
        except Exception as e:
            print(f"Error formatting date '{date_str}': {e}")
            return date_str

In [67]:
def format_sdt(phone_number):
    """Ensures the phone number is a string, contains only digits, and starts with '0'."""
    try:
        # Keep only digits
        phone_str = re.sub(r'\D', '', str(phone_number))
        
        # Add leading '0' if it's missing
        if not phone_str.startswith('0'):
            phone_str = '0' + phone_str
        return phone_str
    except (ValueError, TypeError) as e:
        print(f"Error formatting phone number '{phone_number}': {e}")
        return str(phone_number)

In [68]:
formating_columns = [
    'ho_ten',
    'sdt',
    'ngay_sinh'
]

### 4. Data Cleaning Function

This function cleans the raw data from the CSV file. It standardizes values, removes unwanted characters, and ensures the data is in a consistent format before being processed.

In [69]:
# Create a dictionary mapping field names to their formatting functions
format_ = {
    'ho_ten': format_ho_ten,
    'ngay_sinh': format_ngay_sinh,
    'sdt': format_sdt
}

In [76]:
def clean_data(df):
    """Cleans and standardizes the DataFrame."""
    
    # Clean 'ho_ten' - Strip whitespace
    if 'ho_ten' in df.columns:
        df['ho_ten'] = df['ho_ten'].str.strip()
    
    # Clean 'sdt' (Phone Number) - remove non-digit characters
    if 'sdt' in df.columns:
        df['sdt'] = df['sdt'].astype(str).apply(lambda x: re.sub(r'\D', '', x))
        
    # Clean 'khoa' and 'chuyen_nganh' - remove prefix words like "Khoa", "Ngành", "Chuyên ngành"
    prefix_pattern = r'^(khoa khoa|ngành|chuyên ngành)\s*'
    for col in ['khoa', 'chuyen_nganh']:
        if col in df.columns:
            df[col] = df[col].astype(str).str.replace(prefix_pattern, '', regex=True, case=False).str.strip()
            
    return df

### 5. Dữ liệu Đầu vào và Biến đổi

Hàm `process_mtec_data_v2` thực hiện xử lý dữ liệu chính:
- Đọc dữ liệu từ file CSV, đảm bảo các trường số nguyên vẹn dước dạng chuỗi.
- Đặt tên lại các cột dựa trên hệ thống mapping (chuẩn hóa tên cột không dấu).
- Thực hiện làm sạch cơ bản (sử dụng hàm `clean_data` ở trên).
- Chuyển đổi lựa chọn Ban (vị trí/ban) thành các dấu kiểm dạng Word Checkbox `☒` hoặc `☐`.
- Tạo ma trận đánh giá kỹ năng từ các lựa chọn "Cơ bản", "Trung bình", "Tốt" thành các dạng Checkbox tương ứng.

In [71]:
def process_mtec_data_v2(file_path):
    # 1. Đọc dữ liệu với định dạng String cho các trường số để tránh mất số 0 hoặc bị format khoa học
    df = pd.read_csv(file_path, dtype={
        'Số điện thoại liên hệ': str, 
        'Mã số sinh viên (MSSV)': str
    })
    
    # Định nghĩa ký tự Checkbox (Nếu vẫn lỗi font, hãy đổi sang '[x]' và '[ ]')
    CHECKED = RichText('☒', font='Segoe UI Symbol')
    UNCHECKED = RichText('☐', font='Segoe UI Symbol')

    # 2. Ánh xạ thông tin cơ bản (Sử dụng key không dấu để an toàn cho hệ thống)
    mapping = {
        'Họ và tên đầy đủ': 'ho_ten',
        'Giới tính': 'gioi_tinh',
        'Ngày sinh': 'ngay_sinh',
        'Mã số sinh viên (MSSV)': 'mssv',
        'Khoa bạn đang theo học': 'khoa',
        'Chuyên ngành bạn đang theo học': 'chuyen_nganh',
        'Số điện thoại liên hệ': 'sdt',
        'Email sinh viên hoặc Email cá nhân': 'email',
        'Link Facebook cá nhân': 'link_fb',
        'Mục tiêu khi tham gia CLB là gì': 'muc_tieu',
        'Định hướng phát triển cá nhân của bạn khi tham gia CLB?': 'dinh_huong',
        'Bạn có thể cam kết dành thời gian cho CLB ở mức nào?': 'cam_ket_time'
    }
    df = df.rename(columns={k: v for k, v in mapping.items() if k in df.columns})
    
    df = clean_data(df)
    

    # 3. Xử lý Ban và Chức vụ
    col_ban_raw = 'Bạn muốn tham gia Ban chuyên môn/vị trí nào? (Có thể chọn nhiều)'
    if col_ban_raw in df.columns:
        df['c_ban_cn'] = df[col_ban_raw].apply(lambda x: CHECKED if 'Ban Công nghệ' in str(x) else UNCHECKED)
        df['c_ban_tt'] = df[col_ban_raw].apply(lambda x: CHECKED if 'Ban Truyền thông' in str(x) else UNCHECKED)
        df['c_ban_vh'] = df[col_ban_raw].apply(lambda x: CHECKED if 'Ban Vận hành' in str(x) else UNCHECKED)
        df['c_ban_cnh'] = df[col_ban_raw].apply(lambda x: CHECKED if 'Ban Chủ nhiệm' in str(x) else UNCHECKED)
        df['c_ban_khac'] = UNCHECKED
        
        def extract_vi_tri(text):
            if pd.isna(text): return ''
            positions = re.findall(r'\](.*?)\;', str(text))
            return ', '.join([p.strip() for p in positions])
        df['vi_tri'] = df[col_ban_raw].apply(extract_vi_tri)

    # 4. Xử lý Ma trận Kỹ năng (Chuyên môn & Mềm)
    # Tự động quét các cột kỹ năng dựa trên từ khóa để mapping chính xác
    skill_keywords = {
        'tk': 'Thiết kế', 'qd': 'Quay dựng', 'ct': 'Content', 
        'fp': 'Fanpage', 'ca': 'Chụp ảnh', 'lt': 'Lập trình', 'mc': 'MC',
        'gt': 'Giao tiếp', 'lvn': 'Làm việc nhóm', 'qltg': 'thời gian',
        'st': 'Sáng tạo', 'gqvd': 'vấn đề', 'thvp': 'văn phòng'
    }

    for key, keyword in skill_keywords.items():
        # Tìm cột chứa từ khóa
        target_cols = [c for c in df.columns if keyword.lower() in c.lower()]
        if target_cols:
            col = target_cols[0]
            df[f'{key}_cb']  = df[col].apply(lambda x: CHECKED if str(x).strip() == 'Cơ bản' else UNCHECKED)
            df[f'{key}_tb']  = df[col].apply(lambda x: CHECKED if str(x).strip() == 'Trung bình' else UNCHECKED)
            df[f'{key}_tot'] = df[col].apply(lambda x: CHECKED if str(x).strip() == 'Tốt' else UNCHECKED)

            df[f'{key}'] = df[col].apply(
                lambda x: CHECKED if str(x).strip() in ['Cơ bản', 'Trung bình', 'Tốt'] else UNCHECKED
            )

    return df.fillna("")



# Thực thi
# df_final = process_mtec_data_v2("ĐƠN ĐĂNG KÝ THÀNH VIÊN GEN 1 - MTEC CLUB(1-2).xlsx - Sheet1.csv")
# generate_profiles(df_final, "template_hoso.docx", "Hồ sơ Gen 1")


In [72]:
def generate_profiles(df, template_path, output_folder):
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
    
    doc = DocxTemplate(template_path)
    for _, row in df.iterrows():
        context = row.to_dict()

        
        for format_field in formating_columns:
            if format_field in context:
                context[format_field] = format_[format_field](context[format_field])

        doc.render(context)
        # Đặt tên file khoa học: HOSO_MSSV_HoTen.docx
        filename = f"HOSO_{row['mssv']}_{row['ho_ten']}.docx".replace(" ", "_")
        doc.save(os.path.join(output_folder, filename))

        print(f'Created file: {filename}')

In [75]:
FILE_INPUT = "danh-sach-thanh-vien.csv"
TEMPLATE = "ho-so-thanh-vien-clb.docx"

df_final = process_mtec_data_v2(FILE_INPUT)
generate_profiles(df_final, TEMPLATE, "output_hoso")


Created file: HOSO_2500018535_Nguyễn_Thị_Ngọc_Ngân.docx
Created file: HOSO_2500017768_Hoàng_Thị_Út_Linh.docx
Created file: HOSO_2400004752_Cao_Thị_Thuỳ_Dương.docx
Created file: HOSO_2700000794_Đặng_Thanh_Tuấn.docx
Created file: HOSO_2500011690_Hà_Quốc_Toản.docx
Created file: HOSO_2400000549_Lê_Thị_Ngọc_Nhi.docx
Created file: HOSO_2500015542_Lưu_Băng.docx
Created file: HOSO_2400001905_Ngô_Văn_Minh_Nhựt.docx
Created file: HOSO_2400001871_Nguyễn_Hữu_Chí.docx
Created file: HOSO_2400008831_Nguyễn_Ngọc_Hiền.docx
Created file: HOSO_2400004657_Nguyễn_Thế_An.docx
Created file: HOSO_2400008830_Nguyễn_Thị_Nhi.docx
Created file: HOSO_2400008261_Nguyễn_Thị_Như_Huỳnh.docx
Created file: HOSO_2400001728_Nguyễn_Trần_Gia_Bảo.docx
Created file: HOSO_2500019500_Phạm_Quốc_Bảo.docx
Created file: HOSO_2400000631_Trần_Quốc_Khánh.docx
Created file: HOSO_2400008936_Trần_Quỳnh_Như.docx
Created file: HOSO_2400007081_Trương_Thùy_Ngọc_Thủy.docx
Created file: HOSO_2400003987_Nguyễn_Minh_Trúc.docx
Created file: HOSO_

In [74]:
df_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2 entries, 0 to 1
Data columns (total 95 columns):
 #   Column                                                                                                                         Non-Null Count  Dtype 
---  ------                                                                                                                         --------------  ----- 
 0   ID                                                                                                                             2 non-null      int64 
 1   Start time                                                                                                                     2 non-null      object
 2   Completion time                                                                                                                2 non-null      object
 3   Email                                                                                                                  